In [ ]:
## Code for 10-60 Data

# Coplots linear and cyclized scaled counts via grouping and multindexing (lid, Common_Name (N-->C))
# Fits gaussian curves to every observable peak in the data
# Picks "best" peak and displays retention time of centroid from gaussian fit

# Import packages as easy abbreviations
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
import os
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.integrate import quad
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Draw


# Define input spreadsheet. Define output folder for image files and name for output .xlsx file
excelsheet = "/Users/grant/Library/CloudStorage/GoogleDrive-brodohfasho@gmail.com/Shared drives/DEL Lipophilicity Paper/Data Spreadsheets/10-60_Testset.xlsx"
workbook = "Sheet1"
output_folder = "/Users/grant/Desktop/10-60_Testset_Plots/"
os.makedirs(output_folder, exist_ok=True)
newdata = excelsheet.replace('.xlsx','_RTs.xlsx')


# Define peak picking parameters
# stddev_threshold = max standard deviation allowed for a fitted gaussian. Fit gaussians with standard deviations greater than this value are not plotted
# min_height_threshold_factor = the minimum peak height considered for gaussian fitting as a function of a percentage of the max y value (1 = only the max y peak is fit)
# fit_width = the width of the x-range around a peak to be used for fitting
# minimum_RT = excludes gaussian fitting for any peaks eluting before this RT (min)
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10


# Define gaussian equation
def gaussian(x, amplitude, mean, stddev):
        return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

# Create pandas dataframe from sequencing data
# Define indexes 
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = list()
    plt.figure(figsize=(12, 8))  # Increase figure size for better clarity
    handles = []  # Collect handles for legend
    labels = []  # Collect labels for legend

    for (lid, row) in group_df.iterrows():
        # Define new dataframe of delimited RT_count data
        chrom_df = pd.Series(row["all_datapoints"])

        # Split the comma-delimited values into a new dataframe (time, counts)
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        # Create a DataFrame with the time and count values
        data = pd.DataFrame({'Time': master_df[0].astype(float) / 60, 'Count': master_df[2].astype(float)})

        # Sort the DataFrame based on the 'Time' column
        data = data.sort_values('Time')

        # Extract the sorted time and count values
        x = data['Time'].tolist()
        y = data['Count'].tolist()

        # Find peaks with a minimum height
        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)

        gaussian_means = []

        # Fit and plot Gaussian for each peak
        for peak in peaks:
            # Define a range around the peak to fit the Gaussian
            # Determine the indices for fitting
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue

            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]

            # Ensure there are enough points to fit
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue

            # Initial guess: amplitude, mean, stddev
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])

            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    color = 'navy' if 'DEL-0044' in lid else 'firebrick'  # Set color based on lid
                    label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
                    fit_x_values = np.linspace(min(fit_x)-0.5, max(fit_x)+0.5, 100)
                    w = gaussian(fit_x_values, *popt)
                    gaussian_means.append((popt[1], popt[0]))
                    plt.plot(fit_x_values, w, label=f"{label} Gaussian Fit", color=color, linestyle='--', linewidth=2, alpha=0.8, zorder=2)
            except RuntimeError as e:
                # print(f"Could not fit a Gaussian to the peak at x={x[peak]}: {e}")
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude
        
        plt.annotate(f'Centroid RT ({lid[1]}): {best_mean:.2f}', xy=(best_mean, best_amplitude), xytext=(best_mean+8, best_amplitude),
                     arrowprops=dict(facecolor='black', arrowstyle="->"), fontsize=12)

        if 'DEL-0044' not in lid:
            out_dict[index].append(best_mean)
        else:
            out_dict[index].insert(0, best_mean)

        # Customize the appearance of each line
        color = 'plum' if 'DEL-0044' in lid else 'coral'  # Set color based on lid
        label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
            
        # Plot the data as a scatter plot with a connecting line
        scatter = plt.scatter(x, y, label=label, color=color, alpha=1.0, marker='o', s=14)  # Increase marker size
        line, = plt.plot(x, y, linestyle='-', linewidth=2, color=color, alpha=0.6)  # Increase line width
        
        # Add handles and labels to the lists
        handles.append(line)
        labels.append(label)
    
        # Manually reorder the legend labels to ensure 'Linear' is on top
        sorted_indices = sorted(range(len(labels)), key=lambda k: (labels[k] != 'Linear', k))
        handles = [handles[i] for i in sorted_indices]
        labels = [labels[i] for i in sorted_indices]
   
    # Customize legend and other plot properties
    plt.legend(handles, labels, loc='upper right', fontsize=16, frameon=True, fancybox=True)  # Increase font size and add frame and box to legend
    plt.xlim(0, 65)
    plt.ylim(0, plt.gca().get_ylim()[1])
    plt.tick_params(axis='both', which='major', labelsize=12)  # Increase tick label size
    plt.ticklabel_format(style='plain', axis='y')
    plt.xlabel('Time (min)', fontsize=16)  # Increase font size for labels
    plt.ylabel('Scaled_counts', fontsize=16)
    plt.title(f"{index}", fontsize=18)  # Increase font size for title
    plt.tight_layout()
    plot_filename = os.path.join(output_folder, f'{index}.png')
    plt.savefig(plot_filename, dpi=100)
    plt.close()

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0:'Linear RT (min)',1:'Cyclized RT (min)'},inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k:[x,y,y-x] for k, (x,y) in out_dict.items()}
Linear_RTs = list()
Cyclized_RTs = list()
deltaRT = list()
for ix0,ix1 in df.index: 
    try:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
      
    except KeyError:
        print(f"DIDN'T FIND {ix0},{ix1}. setting to 'np.nan'")
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)
df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False)

In [7]:
## Code for 10-40 data - Sorts the data based on Time - Set Max Count Threshold Below (Unique to 10-40)

# Coplots linear and cyclized scaled counts via grouping and multindexing (lid, Common_Name (N-->C))
# Fits gaussian curves to every observable peak in the data
# Picks "best" peak and displays retention time of centroid from gaussian fit

# Import packages as easy abbreviations
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
import os
%matplotlib inline
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.integrate import quad
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import Draw

# Define input spreadsheet. Define output folder for image files and name for output .xlsx file
excelsheet = "/Users/grant/Library/CloudStorage/GoogleDrive-brodohfasho@gmail.com/Shared drives/DEL Lipophilicity Paper/Data Spreadsheets/10-40_Testset.xlsx"
workbook = "Sheet1"
output_folder = "/Users/grant/Desktop/10-40_Testset_Plots/"
os.makedirs(output_folder, exist_ok=True)
newdata = excelsheet.replace('.xlsx','_RTs.xlsx')

# Define peak picking parameters
stddev_threshold = 2
min_height_threshold_factor = 0.35
fit_width = 1.5
minimum_RT = 10

# Define gaussian equation
def gaussian(x, amplitude, mean, stddev):
    return amplitude * np.exp(-((x - mean) / (2 * stddev))**2)

# Create pandas dataframe from sequencing data
df = pd.read_excel(excelsheet, sheet_name=workbook)
df.set_index(["Common_Name (N-->C)", "lid"], inplace=True)

# Group by the first level of the MultiIndex
grouped_df = df.groupby(level=0)

# Create a dictionary for RT data collection
out_dict = {}

# Iterate over grouped data
for index, group_df in tqdm(grouped_df):
    out_dict[index] = []
    plt.figure(figsize=(12, 8))
    handles = []
    labels = []

    for (lid, row) in group_df.iterrows():
        # Check if the "max_count" value is less than 30
        if row["max_count"] < 30:
            continue

        chrom_df = pd.Series(row["all_datapoints"])
        chromvalues_df = chrom_df.str.split(",", expand=True)
        chromvalues_new = chromvalues_df.transpose()
        master_df = chromvalues_new[0].str.split("[:,;]", expand=True)

        data = pd.DataFrame({'Time': master_df[0].astype(float) / 60, 'Count': master_df[2].astype(float)})
        data = data.sort_values('Time')
        x = data['Time'].tolist()
        y = data['Count'].tolist()

        peaks, _ = find_peaks(y, height=max(y) * min_height_threshold_factor)
        gaussian_means = []

        for peak in peaks:
            indices = [i for i, val in enumerate(x) if x[peak] - fit_width < val < x[peak] + fit_width]
            if not indices:
                continue
            fit_x = [x[i] for i in indices]
            fit_y = [y[i] for i in indices]
            if len(fit_x) < 3:
                continue
            if x[peak] < minimum_RT:
                continue
            initial_guess = [max(fit_y), x[peak], np.std(fit_x)/2]
            bounds = ([0, x[peak] - fit_width, 0], [np.inf, x[peak] + fit_width, np.inf])
            try:
                popt, pcov = curve_fit(gaussian, fit_x, fit_y, p0=initial_guess, bounds=bounds, maxfev=1000)
                if popt[2] < stddev_threshold:
                    color = 'navy' if 'DEL-0044' in lid else 'firebrick'
                    label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
                    fit_x_values = np.linspace(min(fit_x)-0.5, max(fit_x)+0.5, 100)
                    w = gaussian(fit_x_values, *popt)
                    gaussian_means.append((popt[1], popt[0]))
                    plt.plot(fit_x_values, w, label=f"{label} Gaussian Fit", color=color, linestyle='--', linewidth=2, alpha=0.8, zorder=2)
            except RuntimeError:
                pass

        best_mean = minimum_RT
        best_amplitude = 0
        for mean, amplitude in gaussian_means:
            if mean > best_mean:
                best_mean = mean
                best_amplitude = amplitude

        plt.annotate(f'Centroid RT ({lid[1]}): {best_mean:.2f}', xy=(best_mean, best_amplitude), xytext=(best_mean+8, best_amplitude),
                     arrowprops=dict(facecolor='black', arrowstyle="->"), fontsize=12)

        # Ensure out_dict[index] always has two elements (Linear, Cyclized)
        if 'DEL-0044' in lid:
            # Linear
            if len(out_dict[index]) == 0:
                out_dict[index].append(best_mean)
            else:
                out_dict[index][0] = best_mean
        elif 'DEL-0045' in lid:
            # Cyclized
            if len(out_dict[index]) == 0:
                out_dict[index].append(np.nan)  # Linear placeholder
                out_dict[index].append(best_mean)
            elif len(out_dict[index]) == 1:
                out_dict[index].append(best_mean)
            else:
                out_dict[index][1] = best_mean

        color = 'plum' if 'DEL-0044' in lid else 'coral'
        label = 'Linear' if 'DEL-0044' in lid else 'Cyclized'
        scatter = plt.scatter(x, y, label=label, color=color, alpha=1.0, marker='o', s=14)
        line, = plt.plot(x, y, linestyle='-', linewidth=2, color=color, alpha=0.6)
        handles.append(line)
        labels.append(label)

        sorted_indices = sorted(range(len(labels)), key=lambda k: (labels[k] != 'Linear', k))
        handles = [handles[i] for i in sorted_indices]
        labels = [labels[i] for i in sorted_indices]

    plt.legend(handles, labels, loc='upper right', fontsize=16, frameon=True, fancybox=True)
    plt.xlim(0, 65)
    plt.ylim(0, plt.gca().get_ylim()[1])
    plt.tick_params(axis='both', which='major', labelsize=12)
    plt.ticklabel_format(style='plain', axis='y')
    plt.xlabel('Time (min)', fontsize=16)
    plt.ylabel('Scaled_counts', fontsize=16)
    plt.title(f"{index}", fontsize=18)
    plt.tight_layout()
    plot_filename = os.path.join(output_folder, f'{index}.png')
    plt.savefig(plot_filename, dpi=100)
    plt.close()

# Ensure all lists in out_dict have length 2 (Linear, Cyclized)
for k, v in out_dict.items():
    if len(v) == 0:
        out_dict[k] = [np.nan, np.nan]
    elif len(v) == 1:
        if 'DEL-0044' in df.loc[k].index[0]:
            out_dict[k].append(np.nan)
        else:
            out_dict[k].insert(0, np.nan)

outDF = pd.DataFrame(out_dict).T
outDF.rename(columns={0:'Linear RT (min)',1:'Cyclized RT (min)'}, inplace=True)
outDF.index.name = 'Common Name'

out_dict = {k: [x, y, y-x] for k, (x, y) in out_dict.items()}
Linear_RTs = []
Cyclized_RTs = []
deltaRT = []
for ix0, ix1 in df.index:
    try:
        if 'DEL-0044' in ix1:
            Linear_RTs.append(out_dict[ix0][0])
            Cyclized_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
        elif 'DEL-0045' in ix1:
            Cyclized_RTs.append(out_dict[ix0][1])
            Linear_RTs.append(np.nan)
            deltaRT.append(out_dict[ix0][2])
    except KeyError:
        print(f"DIDN'T FIND {ix0},{ix1}. setting to 'np.nan'")
        Linear_RTs.append(np.nan)
        Cyclized_RTs.append(np.nan)
        deltaRT.append(np.nan)
df['Linear RT (min)'] = Linear_RTs
df['Cyclized RT (min)'] = Cyclized_RTs
df['Delta RT (min) (Cyclized-Linear)'] = deltaRT

df.to_excel(newdata, merge_cells=False)

  0%|          | 0/2 [00:00<?, ?it/s]